1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working')
REPO = ROOT / 'heartlib'

'''
Select the downloaded HeartMuLa and codec checkpoint directory.
設定 HeartMuLa 主模型與 Codec 權重的下載位置。
'''
CKPT = Path('/tmp/heartmula/ckpt')
HF_CACHE_DIR = Path('/tmp/heartmula/huggingface')
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Select the optional remote FIFO API port; change only if the port conflicts.
指定選用 FIFO 遠端 API 的本機連接埠；只有發生 Port 衝突時才需要修改。
'''
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_HeartMuLa_音樂生成_Kaggle'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID

'''
Limit how many remote jobs wait for the single GPU worker.
設定 FIFO 最多等待的工作數量；超出時回傳 HTTP 429，client 會延遲重試。
'''
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2

'''
Mark exceptionally long remote inference as stalled, without forcibly starting a second GPU job.
設定遠端推論超時警戒值；卡住時不強行啟動第二筆 GPU 推論。
'''
REMOTE_STALL_HOURS = 2
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝與 GPU 檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch
assert torch.cuda.device_count() == 2, '請在 Kaggle Settings 選擇 GPU T4 ×2。'
CKPT.mkdir(parents=True, exist_ok=True)

'''
Keep the Hugging Face cache on temporary storage.
將 Hugging Face 快取放到暫存空間。
'''
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_XET_CACHE'] = '/tmp/heartmula/huggingface/xet'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

'''
Download the official source if it is not already available.
只有尚未存在官方原始碼時才下載。
'''
if not (REPO / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HeartMuLa/heartlib.git', str(REPO)], check=True)
else:
    assert (REPO / '.git').exists(), 'HeartMuLa 路徑不是 Git 倉庫，請先檢查。'
    changes = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()

    '''
    Update only when doing so will not overwrite customized files.
    只有不會覆寫使用者修改時，才同步上游程式。
    '''
    if not changes:
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', '-q'], check=True)
    else:
        print('HeartMuLa 原始碼已有本機修改，為避免衝突未自動更新；建議使用全新 Kaggle Session。')

print('HeartMuLa commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.0', 'tokenizers==0.22.1',
    'huggingface-hub==0.36.2', 'vector-quantize-pytorch==1.27.15', 'soundfile',
], check=True)

'''
Keep Kaggle's PyTorch and TorchAO when installing TorchTune.
安裝 TorchTune 時保留 Kaggle 的 PyTorch 與 TorchAO。
'''
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchtune==0.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))

'''
Give installation checks the same import priority in their subprocesses.
讓安裝檢查子行程也優先載入正式套件，而非外層 Git 目錄。
'''
os.environ['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
subprocess.run([
    sys.executable, '-c',
    'import torchao, torchtune, torchaudio; from heartlib import HeartMuLaGenPipeline',
], check=True)
for gpu in range(2):
    print(f'GPU {gpu}: {torch.cuda.get_device_name(gpu)}')

print('✓ 第 2 節完成：環境安裝與雙 GPU 檢查已完成。')


3. 官方模型與 RoPE 相容修正


In [ ]:
'''
Import the official Hugging Face model snapshot downloader.
匯入 Hugging Face 官方模型下載工具。
'''
from huggingface_hub import snapshot_download
for model_id, directory, patterns in [
    ('HeartMuLa/HeartMuLaGen', CKPT, ['tokenizer.json', 'gen_config.json']),
    ('HeartMuLa/HeartMuLa-oss-3B-happy-new-year', CKPT / 'HeartMuLa-oss-3B', ['config.json', '*.safetensors', '*.safetensors.index.json']),
    ('HeartMuLa/HeartCodec-oss-20260123', CKPT / 'HeartCodec-oss', ['config.json', '*.safetensors', '*.safetensors.index.json']),
]:
    snapshot_download(repo_id=model_id, local_dir=str(directory), allow_patterns=patterns, max_workers=2)

model_file = REPO / 'src/heartlib/heartmula/modeling_heartmula.py'

'''
Read the source before applying the known RoPE compatibility patch.
套用已知 RoPE 相容修正前先讀取原始碼。
'''
source = model_file.read_text(encoding='utf-8')
original = '        with device:\n            self.backbone.setup_caches(max_batch_size, dtype)'
replacement = (
    "        with device:\n"
    "            # Restore RoPE caches on the actual compute device.\n"
    "            # 在實際運算裝置上重建 RoPE 快取。\n"
    "            for module in self.modules():\n"
    "                # Repair position embeddings only when their cache is missing.\n"
    "                # 只修復尚未建立快取的位置 embedding。\n"
    "                if hasattr(module, 'rope_init') and not getattr(module, 'is_cache_built', True):\n"
    "                    # Initialize the missing positional cache.\n"
    "                    # 初始化缺少的位置快取。\n"
    "                    module.rope_init()\n"
    "                    # Keep the positional module on the model GPU.\n"
    "                    # 確保位置模組維持在模型所使用的 GPU。\n"
    "                    module.to(device)\n"
    "            self.backbone.setup_caches(max_batch_size, dtype)"
)

'''
Avoid applying the compatibility fix twice.
避免重複套用相容修正。
'''
if 'module.rope_init()' not in source:
    assert original in source, 'HeartMuLa 的 setup_caches 已變更，請檢查官方版本。'
    model_file.write_text(source.replace(original, replacement, 1), encoding='utf-8')

print('✓ 第 3 節完成：模型相容設定已完成。')


4. 載入模型（僅首次）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import sys
from pathlib import Path
import torch
sys.path.insert(0, str(REPO / 'src'))
cached_heartlib = sys.modules.get('heartlib')

'''
Remove a namespace-only import without discarding a valid package.
僅移除錯誤的 namespace 快取，保留已正確匯入的套件。
'''
if cached_heartlib is not None and getattr(cached_heartlib, '__file__', None) is None:
    del sys.modules['heartlib']

from heartlib import HeartMuLaGenPipeline
import heartlib
assert Path(heartlib.__file__).resolve().is_relative_to(REPO / 'src'), f'heartlib 匯入來源錯誤：{heartlib.__file__}'

'''
Assign token generation and audio decoding to separate GPUs.
將 token 生成與音訊解碼分配到不同 GPU。
'''
devices = {'mula': torch.device('cuda:0'), 'codec': torch.device('cuda:1')}
if globals().get('pipe') is None:
    pipe = HeartMuLaGenPipeline.from_pretrained(
        str(CKPT), device=devices,
        dtype={'mula': torch.float16, 'codec': torch.float32},
        version='3B', lazy_load=False,
    )
    print('已首次載入模型；後續生成請執行第 4、5、6 節。')
else:
    '''
    Confirm that the existing pipeline is the intended implementation.
    確認既有 Pipeline 是正確的官方類別。
    '''
    assert isinstance(pipe, HeartMuLaGenPipeline), '既有 pipe 不是 HeartMuLaGenPipeline。'
    print('沿用既有模型，未再次載入權重。')

assert not pipe.lazy_load and pipe._mula is not None and pipe._codec is not None, '雙 GPU 模型未完整載入。'

'''
Verify that all HeartMuLa parameters are on GPU 0.
確認 HeartMuLa 所有參數均位於 GPU 0。
'''
assert {p.device for p in pipe.mula.parameters()} == {devices['mula']}, 'HeartMuLa 並未完整位於 GPU 0。'
assert {p.device for p in pipe.codec.parameters()} == {devices['codec']}, 'HeartCodec 並未完整位於 GPU 1。'
for gpu in range(2):
    print(f'GPU {gpu}：{torch.cuda.memory_allocated(gpu) / 1024**3:.2f} GiB 已分配')

print('✓ 第 4 節完成：音樂模型已載入並完成 GPU 分配設定檢查。')


5. 生成設定


In [ ]:
'''
Define the lyrics for the next generation.
設定下一首音樂使用的歌詞。
'''
LYRICS = '''
[Verse]
十年生死兩茫茫
不思量，卻自難忘
千里孤墳隔月光
無處話淒涼

[Chorus]
夜來幽夢，忽然還鄉
小軒窗前，你正梳妝
相顧無言，淚落千行
一醒只剩，滿地月光

[Bridge]
料得年年最斷腸
明月夜，短松岡

[Chorus]
隔著十年，隔著陰陽
明月仍照，那座松岡
'''.strip()
TAGS = '''
soft,sad,ballad,longing
'''.strip()
DURATION = 20

'''
Choose a fixed seed, or -1 for a random seed on each generation.
設定固定種子，或使用 -1 讓每次生成採用隨機種子。
'''
SEED = 42
OUTPUT_NAME = '十年生死兩茫茫.mp3'
print('✓ 第 5 節完成：歌詞、曲風和生成參數已設定（尚未生成）。')


6. 生成音樂（可重複執行）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import time
    import heartlib.pipelines.music_generation as music_generation
    import random
    import secrets
    import numpy as np
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    random.seed(RUN_SEED)

    '''
    Seed NumPy before sampling.
    取樣前同步設定 NumPy 種子。
    '''
    np.random.seed(RUN_SEED)
    torch.manual_seed(RUN_SEED)
    torch.cuda.manual_seed_all(RUN_SEED)
    print(f'本次生成 seed：{RUN_SEED}')
    lyrics_file = ROOT / 'heartmula_lyrics.txt'

    '''
    Update the input file on every new generation.
    每次生成時更新歌詞檔，確保使用最新內容。
    '''
    lyrics_file.write_text(LYRICS, encoding='utf-8')
    tags_file = ROOT / 'heartmula_tags.txt'
    tags_file.write_text(TAGS, encoding='utf-8')
    filename = Path(OUTPUT_NAME)

    '''
    Reject directory paths and non-MP3 extensions.
    防止輸出到其他資料夾，並限制副檔名為 MP3。
    '''
    assert filename.name == OUTPUT_NAME and filename.suffix.lower() == '.mp3', 'OUTPUT_NAME 必須是單純的 .mp3 檔名。'
    output = ROOT / filename
    index = 2
    while output.exists():
        output = ROOT / f'{filename.stem}_{index:03d}.mp3'
        index += 1
    original_tqdm = music_generation.tqdm

    '''
    Print actual generated-frame progress, even without Notebook widgets.
    即使 Notebook 沒有啟用 widget，也能顯示實際逐幀生成進度。
    '''
    def show_progress(frame_indices):
        '''
        Read the total number of 80 ms frames allowed in this generation.
        取得本次允許生成的 80 毫秒音訊幀數量。
        '''
        total = len(frame_indices)
        interval = max(1, total // 10)
        print(f'音訊生成中：最多 {total} 幀（{DURATION} 秒）', flush=True)
        for frame in frame_indices:
            if frame > 0 and frame % interval == 0:
                print(f'長度進度 {frame / total:.0%}｜約 {frame * 0.08:.1f}/{DURATION} 秒｜耗時 {time.perf_counter() - start:.0f} 秒', flush=True)
            yield frame
        print('長度進度 100%｜已達音訊長度上限', flush=True)
    used_gpus = set()

    '''
    Capture the device of each selected layer's output tensor.
    以指定模型層的輸出 Tensor 裝置確認實際運算位置。
    '''
    def record_gpu(module, inputs, result):
        '''
        Identify the device used by this model operation.
        取得本次模型運算實際使用的 GPU 編號。
        '''
        gpu = result.device.index
        if gpu == 1 and gpu not in used_gpus:
            print('[2/3] GPU 1 正在解碼音訊...', flush=True)
        used_gpus.add(gpu)
    mula_hook = pipe.mula.codebook0_head.register_forward_hook(record_gpu)
    codec_hook = pipe.codec.decoder.scalar_model.decoder[0].register_forward_hook(record_gpu)
    start = time.perf_counter()
    print(f'[1/3] 正在生成 {output.name}，音訊長度上限 {DURATION} 秒...', flush=True)

    '''
    Use text progress for this run only, without modifying model weights.
    只在本次執行使用文字進度顯示，不修改模型權重。
    '''
    music_generation.tqdm = show_progress
    try:
        with torch.no_grad():
            pipe(
                {'lyrics': str(lyrics_file), 'tags': str(tags_file)},
                save_path=str(output),
                max_audio_length_ms=DURATION * 1000,
                temperature=1.0, topk=50, cfg_scale=1.5,
            )
    finally:
        music_generation.tqdm = original_tqdm

        '''
        Remove the GPU 0 monitoring hook after this run.
        本次執行結束後移除 GPU 0 的監測 hook。
        '''
        mula_hook.remove()
        codec_hook.remove()
    print('[3/3] 已完成音訊生成與儲存，正在檢查結果...', flush=True)
    assert used_gpus == {0, 1}, f'雙 GPU 運算驗證失敗：{used_gpus}'

    '''
    Require the generated MP3 to exist and contain nonempty audio data.
    確認本次成功產生非空的 MP3 音訊檔案。
    '''
    assert output.is_file() and output.stat().st_size > 1024, '找不到有效的 MP3。'
    for gpu in range(2):
        print(f'GPU {gpu}：已驗證運算；VRAM {torch.cuda.memory_allocated(gpu) / 1024**3:.2f} GiB')
    print(f'MP3：{output}；生成耗時：{(time.perf_counter() - start) / 60:.2f} 分鐘')

print('✓ 第 6 節完成：MP3 已生成並通過檔案檢查。')


7. 試聽與下載


In [ ]:
'''
Import notebook audio playback and download links.
匯入 Notebook 音訊播放與下載連結工具。
'''
from IPython.display import Audio, FileLink, display
display(Audio(filename=str(output)))
display(FileLink(str(output)))
print('✓ 第 7 節完成：試聽和下載連結已顯示。')


8. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'music'
REMOTE_FIELDS = ['task', 'lyrics', 'tags', 'duration', 'seed']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xadZ\xefS\x13Y\xd6\xfe\xce_\xd1_\xa6\x12\\\x89\x803\xfb\xce\xb8\xcbV\xf9\xceLMY\xbb\xa3\x94\xa3[\xf5\xd6\xd4V\xaaIn\x92~\'\xe9\xcevw@^\x8a*pd\x00\x01\xa1v\x00\xe5\x87\xa2\x88\x1a\x7f\x00\xea\xa0"\x82\xfe1\x9b\xee$\x9f\xfc\x17\xf69\xf7\xdc\xeet\x02\xb8\xfb\xe1\xf5\x8b\xa4\xd3\xb7\xefs\xce=\xe7<\xcf9\x9dX,\xd6v\xaeP\xb4lWssB\xcb\x1b}\xb6n\x1b\xc2\xd1L!\xd2"\xade,\x1b_\x18\x8e\xe6\x88\x94kXf\xa2\xcd\x9b\xde\xf6\xc6\x1e\xf8\xab\xcf\xfc\xadW\xd5\xf2\x94\xb7?\xebO\x8e\xd4WG\xaa\xcb\xd7\xbc\xf1\x03|\xf4\xf6\x9e\xfes\xe4j[\x0cO6\xd4\x93\x8d\x82\x08\xfe\xce\t\xddv\xb1M\xa2h\x14E\xde0\x85\x93(\x94\x1c#\x95\xcc\nS\xd8:\xed\xa1\xe9\x8e\xd6z-Xn\xebf\xda*\x04\x9f\x00\xca\x16\xae\x13|4K\x85\xe2 \xad6\x8bmr\xff\x8b\xc2\xb1\xf2\xfdB\xd35[\x14m+]J\x19}y\x81e"\xfd\x07\xad\xa3KS\xcf\x87\xb9:\x0c\x1e\x90_4L\xb6K0\xb7r\xf0\xa1:_\xf6V\xf6\xbc\xad\xe5jy\xcb\xdb\x9c\xab\x8f\xcfTg\xdfW_\xcd\xf9wV?\xee\xaf\xe01\xde\xe4x\xf5\xea\x1e\xb9\xe4\xd9\xbd\xea\xfc\x9a?1W\x9d_\xa7?\x16\x9f\xf3\x92\xc0\x1f\x17/\x9fO\xfe\xf0\xed\xb7\xdfh=\x01\xf4\x04\x19\xd4\'\xf2\xd6@\xbc\xfb\xc4\x89\xd3\xdd\xed\x9a\x91\xd1\xf8\x96\x1eB(\xf2\x8e\xd0\x0c\xd3\x8d\xd3\xb5v\xb6\xea\xcfB\x14\xe5iI\xb8\x03\x06\xc0\x9a\xfc9\xa7\xdb\xb8r\xba\xbb\xa3\xcf\x90\xae\xca\n\xcd\xcah\xe7K\x85^\xb8\xc5Lk\xbd\x83\x97,;\x95K\xb4)K\x96\xe6\xbc\x89\xd7\xdejY\xddR\x9b\x18\x0fn\xd1\xbc\xdb\xe3\xfe\xfc\xb6?\xbb\x83s\r\x9eX\xdd\xbe\xe6\xad\xce\x04\xc6\xe8\x8e#\xe0\xf4N\xed\x8f=Zh\xd8\x1f5i\xc6I-&?z\x1f\xc6\xeaw\'*\xef\xa6\xfc\xc5}\xdcY\x1b\xffM\xfb\xbc\xfb\xab\xcf\xbf\xfa\xfd\x7fu\x7f\xf5\xc5\xc7\xfdi\x7fb\xb1V\xde\x84\xf3`+\x9e\x1bc\x03\x7f \xbb\xf4|>\x12\x8e}\x02\xc7\x02\x0b\xf5B\x11A\x93\x95V[%\x18)\xfaJF>M\x97d\xfcZ:Em\xc1J\xc3q\x896\xffF\xb9\xf6t\xd1\x9b]\xf4\xcb\xf7\xbd\xc9\x19l\x85S\xac/\x97\xfd\xc7k\xec\x01@\xa8\xec\xce\xe0DqT\xde\xbb\xbd\xea\xd3)\xef\xcd\xcb\xda\xfe$"\x1cv\xfb\xe5{\xde\x9d\xa9\xc0^\x0e\xbd\x049=\x1e\xd8\xdb\x1e\x01\xcc>l\x01\x8a|\tv\xf7\xe6\xa6\xfd\xcd\x07\x8cA\xdd\xdc\x1c\x1df1\xf1\xef\xf6 \x1b\x0b:\x8e;8&^\x10\x04\xb2e\'\xda\xd4\x06\xc1\r0\xa3\xb2\xfb\xae\xf6p\x94\xcd\xf6\x17v96\xbd\xa5r\xf3\xee\xae\x8c\x8c\x82n\x96\xf4|\xf2X\x00\xad\xfb9\x1a\xe5+\x8eJ\xef\xd7\x8d\xbcN\xd9\xf5]\xefe\xb8>j-\n\x84\xbf:\xe9\xcdn#\x93\xe8k\x02\xf5\x9f\xc1I\x95\xd2z\x14S\x12[\x1d\xc2\x95\xb3\x06\xa4cD&CE\xaa_4\xf2\x18i/tW\xc2\x92\x15\xa5#Re\x90\xf6\xc8\xbf\xb6\xfa\xbd\xed\xea\xc6\x9e\xb7\xfd\xb0\xbe\xbc\x12\x06\x85\xbf\xf8\xb6\xf2\xfe\x83\xb7\ttk\x9c\xef\xb0\xa7\xb2;R}u\xad\xbe\xf6\x9b_\xbe\xcai\xce\xe6\x05\x90\x8b6%j&\x16-\x04\x12\xca\xc7\xfd\xe5\xa1\x00\xf4p,\x80\xad\x03\xa8\x8cY*@\xae\x96\x1f\xb4\x8d\x14@]A\xbd\xb4P\x8f2\x06@\xeb\xa9\x94(\xba0\xa6oP\xdeke2F\xca\xd0\xf3ZP=\xe1\xe8\xe7\xe3\xfe\xea\x08\xe2\xd7\xdf\x9c\xae=\xbe\xe3m?E\xf0z[\xb7`\x82\xd6\xab\xee\xd2\xe0\xfb\xda\xd6\x08B\x91\xa2zq\xdc\xdb\xbc\xe9?\x99\x0fp\xcb\x9d\x9d\xa4\xdc\x10\xa9|\xe1\xc2%\xed\x94\x16\x93\x85\xbaP\xca\xebI\xfe>\xe1^qU~^.\xa6\x81Y\x022\xcc"\xb2P.\x85KE\xbf\xb0\x07e%m\xf8\x19Y\xb8=\x1b\xfa\xc3_\xba\xea\xaf\xec\x84h\x81\x02\xde\xae\xae\xefU>\xdc\xe6J\xcb\xb6xc\x8f\xbc\xad\xb7G\x00L\x0c\xd8\x86+\x92\xe4\xa6\xf8_\xfe\xe7\xe2\xb9\xaf\x7f8\xa9\t3eQ\x05\xe8\x89\x95\xdcL\xc7\x97\xc79\xd8q\x07\x812-\x9c\x94m\x149\x02,\xcdp\x9d\x88\rQo\xae\xbc\xac\xaf\x97\xfd\xf2R\xf5\xc5\x06\xfb\xb4\xb6\xbfK\xc4\xf7d\xde\xbf7\x11F\xa9\x9e\xfd\x84\xe3\xe8\xdb\xc3nc\x1c\xf4]P,\x84\x8eD=\xcec\x95\xb7S(\x1e\xcai\x11P\x87 D]s\xe9\xecw\xc7;\xe6\x9c\xe9\n\xbb\x08\xf6\x91\xde\xa1\xa5\xa6^\x10Z\xca23F\xb6D\x0c\xa2\xf7Y\xc4\x9aD\x8by+\x85p\xfb\xbe\xf7tx\xa3t\x92J\xec\xbb\xfb\xfe\xf481#\xc2\xea\xc9\xbc77S{\xb8\xa8\xa8\xf0\xf1\x9a\\\xc5\x97\x03\xb0\xe1f=Z\xaf\xee\xe6\xe2\x17._\xea\xbd|)y\xfe\xec\xf7\xdf\xb6\x07\x94\xfd\xbfHb-m\xd8\xf8\xcfB4\x15q\x9f#\xd9\xcb\xb4\xcc\x0ez$\x0c\x14\xa6\x03GQ\xf6\xdez\xe9o\xae\xd3\xd1\x8c\xefy\x13\x88\x9b\xd7\x95w\x8b\xb5\xdf\xc6\xfd\xc5%o\xe3\xbd,\xef\x0f\x15\xcdMn3\x16b\x1b<\xa6\x85\xc6B\xeb\x18^\x8f\x16\x81&w\x0fopJH\xc2+\tP\xb6\xb0\xe3\xedtk,Q(\x9e\x8e\x81\xf3\xa2k\x98\xfa\xfc[\xdb\xde\xe2Vug\x9eh\x94nk8$\x88\n\x87\xa34\xe2\xfe\xd0K,E\xe8\x8cl\x04p4B\xaa\xcf\x9e\xa1\x1c5\xf4\x06\xc2\xe3\xc6\x03\x95A\xcd\'\x12\xd8\t\xb6\xa40\x0f\xe34\xd8D%\x8b\xab\xb3\x80\xea\x136\x91\xa9\x0eb\xef\xecFN\xe7\x07\xb5\x81\x9c`}\x81{\xb2Be\xbb\x9e\xb7\x85\x9e\x1e\xc4a\x18\xb2\x8cz\xb3OP\xe3yo\xce\x110\xa9\xb7y\x0b\xea\x029O%\x15\x0c\xf8\xfe\x8e|j}q\xca{4\xe5]\xbf[\xd9\xbd\xee\xbf\x1a\xf5\x9f\xef\xd4\x96\xd6B\xcdh\xa6\xc5\x15\xc0\xecf`\xbd\xb6\xc0\xf9 \x1a\x11\xb0\xfd\x86UR\xd2\x90\n\xa2\x9e\x96\xbc\xafk|"\x94\xce\\\xef#.d&\x82\x1b\x0ec\xaa\x8d?\xf1\xa6\x16\x18E\xf5M\x19\x10\x80\xb2>\xfa\xc1\x1bC\x14\xffR\xfbu\xca\x1b\x9b@\xe6\x11W\xc9\x92\x1f\x00\x1c\xc8\xc9j\'\xdd\x99`\xf3\xe3\xedg\xda\xda4\xfc\xa3\xef\xe9\xff\xafs\x96\xa5\xce\xd5\xa4\x8a\xde`Gv1\x106\xa7\x14-\xaa\xbc\xbf\xed\xed\xcdz\xcf\xa7\xfc\xc9\xf7\x95\xdd)\x9c\xaf72\xe5=_\xf6W\x9f\xf0\xc9R\x0c5\xe7Tt\xcfC\x07\x1c\x1bj\xc4\xac+\n\xc3\xc9!\xe9\xdb3\x9d\xa7\xd3\xc32b\x9b1\x7fO9\x0f\x1f\x86\xa0\x19*\tR\xdd\xb4p\xd5\xe6\x93\xcf\xa1.H\xb5I\x11\xda@_\x9bzP{\xfc\x92\x1dYy\xbb\x06O3h\xaa\xefo\xf7\xab{\xd3-\x86\xf1\x9d\xadF\xf0\xe9\xff\xaeG\xebj\xd1\xba\xa5\xa2\xe3"\xe2\n\x88\x03+\x8b\x98@\xd9\x86\xc2\x12\x05a\xbaL\xe7L\xf7\x0ej\x86j"2\xa8r\x11!\xef/?\xa5\xa3\x97\xc4X\x1fy\xe9\xed=D\x89B]\x90l\xbf\xcf\x9c\xed\xad\xbd\xa9\xdd\x9b&i\x7f{\xcd{?]\xbfz\xcd\xbb\x11\x06\xa6e\x1bY\xc3\x84\x10q\xff\x9e.\xc0\xcd\xad-J\x82\xae\x07A\x0b%\x00\xf2v\xa1[\xc2&#\xdd\x91\xb1\xc9e\x81\x01\'\x89.\xcdP\xc9\x9e\xb7\\\xd1gY?\xe1B:+dJ\xcd\xfc\x06\x176\xbe\xf0_\xfe\x83d\xd4\xc2\x1a\xc9(\xbe\x8b\n\xdc\xdb\xb5\xda\xcf\x07Q\x15S\x1f\x99\xf3\xde\x8epa`K\x03\x13\xd2"\x83&\xc1\x1aH\x06\x18\xe2\x12R\x12^7R\xc2i\x8d\xe2\x8b\xc8pN|\xcb\x85!*\x1e\xd0Q|\xd9\xa9\x15\x1cM\xaeuH\xfd\xa1\x14\xa65\xd9\x84\xc0\xdb\xd12EO\x81\xea\xf0\xde\xdfT\x0e\x1e\x1b\xad\x95_\xa8\x9a\x85h\xc6\x83\xfc\xed\xa7\xd5G\xff@\x8a\xd5\xca\xd7\x01\x1b\x8a\xb0>>\xdb\x1a\x16\x0c\xa0GC8\xb7@nE\xcc]\xa2U\xca\xe6P\xb9X\x8ftu~&%U\xbfe\xa4Q\xaf\xb0\xca!\x91\x98\xb7\xb2\x1a7\xa0\x0cs\xe3Qu\xe7\x1a\x98W\xde\xef\xad\xdc\xf1\xee\xbe\xe0\x12\xdb(\x0b\x92f\xea\xa37\xbc\x8d\xe5\xea\xceH}\xfa\xda\xe1\xf0E\xd0\xf5K\xa8\x05\xfdJ\xbc\xeb\xa4B~\xea\x14\x9e\xda\x82\xf5\x1cY\xc0:\n\xd5VZ\xd5\xd17\xa8\x82$\xa2S)\xdf\xfaD\xb6\xa4\xdc\xa9N\xfa\xcdK\xae\xa2\xc1a\x93\x88f\'\xb6b\nd)\x7f\xabD\xc5\xee&D)\x94\x0e,\xd1\x86$\xc6a\r\xcf\xf9\xb8?1\xf4\xcd\xe5\x8bg/\x9d\xbbp~X\xc3\xb9|\xdc\x9f\x04\xb1e\xf2%\'\xd7s\xc9.\x89\x16\x1bz\xc1\x9e,_\x82\xec`CT\x1e\xabj\x12\xaa\xd6\x88Uy\xcb**\xc7Crm\xcfR\xa9\xbb\xb1F\xb4\x00kv\xd6\xbd\xfd\x85\xca\xde\x06\x9d\x82LX\xc4\x8aR\xdc\x1fv\xbc\xd5\x89V\x13)\xf7\x83}\xb5\xa6\xf8P\x11\x1d\xbd9\x12\')\x8bj\x88\xe4\x0b\x15\xcb\xd4Qf\xa1\xdd\xec\xf0(UxH\xa4\xa0\xb6\xd55\x0e\rb\x92\xadi\x0e\xe3c\x82\xb7uS\x14R\x06\xf9\'\xb4\xc2RR\xc8O\x9fE\xa2\xa6G\xeb\x8c n}\x00\xfd\x0b\xbb\x9dFYI\x97\x02\x9f\x1a\x05CFR\x8a+ \xb6\x18@vv\xa4 \xe2~\x92S\x98D\xd3\xb38\x94T\xa1Xx\x83B\x01*\x84Z"\x93d\xdd\xa0\xf2\xf2\xee\x817{]\xb5G\xaf\x7fE\xf0\x83:\xeb\x8b\xbfF\x8d<\ng\x18u\xf2\xb9\xfc8m\x88->\xc5Yq&\xd1\xf9\xd9\xf0\xc7\x83U\xa4]\xf0\xcd\t\xad3\xd1\xf9\xe5\x99DWf\xf8TK \x1e\xac\xd6Fnbo\x84+\x19R\x14v&\x99\xb2J\xe4:\xa8\xb0\x0e\xc8j(\x18<2#o?*h[aJn1\xcc\x0c\xe8\xd8L\t\xe4XN\x87\xc2\xb0\xb5\x92\x99\xca\xd1\xec$\xdd\xf0\x16\xda\x13\x7fz\x94c\x91\x87\n\xc4\x13W\xf7*\xbb3\xb5\xad\xeb\xc7\x1d\xf8\xa0!\xf2\xea\x94\x8f,S]\x9d\xa84Rb!4\xa4\xce+\xa1\x93\x8e\nA\xe9\x14>V\xe21\xca\xb4\x00\x96\xf7\xf3\x18t\x0c\xc5\xe0\xc2N}t\x1e\xe2\x975yK<\xf2\x89\xd2\x99\xc9\xc3\x96{\x1e]!\x9a\x8f\x8a\xee\x83\xcf\xa9\xce\x8c\xce\xf3\x03\xa3!\xd2\xea_\x96\xee)\xcbF\xc8\xe5\x0c\xd4\x83\xaf/\x7fs\x16\xcdV?%\xa1bB*\xc8WD\xaa\xa4ZGG\xe4!\xf1\x83i\r\xba\xb4Aa;4\xbf\xb8\x85\xc2\xaa\xe8B\x86\x1d\xb3\xb2\x92\x96r$\xe3\xbd\xd8 \xf6\x90{\xd4\xee\xdf\xae\x1el\x05\x0cWrD:\x99-B&\xd2\x88\xcd\x8d+h_\xebE\x17\x1e\x95\xfb2(\xa21Y\xb8B\x18\x12@\xcc\tt\x14\xf5\x184OA\x02\x1c\xde\x9a\x99@\xbb$\xefQ\x10\xd0\xc5\xd6\x9e\xcc(\n\x1e\x9d\xaan\xdd\xac\x1c\xccD\xa0\x11\xf9\xda\xd2E\x040\x0e\xabKyq\x92\xbbO\x88\x01\xd0q)\xef\xb6R\xf0\xb94\x04\x8e\x91\x19\x8c"\'\x1by \x80\x98`\xe7Y\xc5c)\x97a3 \x06\xd7\x10\x93r\x16s\xa4\x0e\x03Bx\x901%x\xdf\x84\xac\xe9\xcd\xf0\xce\x9a&R\x10\x98\xf4R\xda\xb0\x00\x90\x1b\xceF\xcb@;t\xf1u\xa9\x1f\xa9\x8f\x81\x1cSE\xb5\xba\xf0Z\xddA>}t\xbf\xba\xbe\xef-\x95\xeb\x0f\x17C-\x16\x06.\x7f[\x9d\x1a\xf1\xc7\xa6\x0e\xb1n\x86\xf1B5\xca\xa2G\x1f\xa0X\x89\x0c\xc2x8\x86\x08~`Q\xc9\xdee\xf0\x1d\xa8$Y\xa1\x81\x0e\x1c\xfa\x1f\xa9a\xb1\xabS"\x9f\x8f\x90\x01\x9a\x088\xf8\xee\xbej>gF\xa0\xd7B\xb8\x9c1\x0c\x1a\x7f\xf8\xa3\xdb\xc7\x15\t\x95|?v\x9f:\xfd7\xe5\x0c\x7f\xf3>\x9e\xce\x8b\xf9A\x89D\xe2S4|6\x95*\xd1\xc4\xc1e\x87+\xee\xd3\xac>\xd9:\xa550\x04OH\x8f\x90g\xd1|ST0z\x83\x06\xfb\x91\xdc\xe3\xe8\xf9t\xc4\x84\x9eN\xa0/\x8b\xe3\x0f\x95z\x17\x18\x04J\xfeO\xc2\x8c\xce\xdf\x88|K,\xde\xe5\x14\x91\x1e\xdd\x89\xcev\xe5\xbe\xbf\xfb\x8c?i\xd4\x9aaW\xb9V\x0b\xd0\x11\x96 \xa9\xe4\x9c%G\xea\xb8G\xce\xc4\x12t!A\xd1F\x92\xb93\x99\x83\x84M\x80\xd2\xd1\xa5\t;\t\x9d0\xa0#\xffhA\xbc\x91\x8b-H\x07\xf4~\x9a\xcb\x14:\x82\xb0=\niW\x13\xd2.\x854z\xec\xcd@\xe9I\xa9&\xa4\xf2JB\xed\x91pR:TGR\xa6sp\xf1\xc7\xce\xbf\xfd\xa7\xd8\xb9\x7f\x073\x06\x93\xf0T\xc9\x06\xadE\x87\x06\xdc\x06\xb1j\xac\x95\'\x80\x8d\x0f=:\xd0\xf4\'Gj\xd7_\xd4^\xbe\xa3\xf1A\xc8\xf3\xb4\x81\xa4W@?\x82|\x83\xea\x1f\xbeE\xa2\x01\xa0\xaa\xa2r\x8c\xa1\xc6\\\xb4\xb4\x03\xec\xe6\x94$J\xae\\!\xff\xa2\xdfA\xccK\x8a\x0fG\xefj~?9\x03!\x82n\x9cS+\x8a\x9as\xafy\xd0\x11(\x8f\x1f\xbb(\xa38\x97\xd40vHu\xed\x84j\x98d\xfd!^\xd3\x9aU\xc7\x11y\x17\xcel\xa8E\x0e\x9b\xd1\xe8K#I\xe9\'\xc3\xc6\x0ev\xa2\x807,\x1e\x10F6\xa7F&\xaa\x88\x04\xd5N\rp\xe4\x80\x96y\x98M\xe6\xd7\x14\x95\x0f[\xfe\xfc[\xe5\x99g\xd7\xeb\xe33\x8d,8\xa2\x15%\x06\x8c\xf6z\x0c\xfd[\xb8\x1fgQ\xb0L\x03\xad2\xa1\xa2\x80\xe2\x01[c\xba\x1c\x98\xa5\xe3Vn\xaaQE\xa8\xc9\xd6\x8d<\x96\x07\xbd\xa9:\x9f\x8d\x17\xfe\xc2M\xd5\x8a>\x1c\xe5\xae\xf9p\xa3]\xd9}X}\xf4\xae\xbe\xb4\xa1\xf2\x86\xf6\r\x87\x98\xf6`\x0b\xf1}c8r^\x92\xb5\xf5\xb4\x81@v\x1a-\\\xc94\x055q:z\xbb\x86|+\x88\x02\x8d\x0bQ\x86\x14\x07\x8e\xae\x92/\xd7\xb7\x01\x80\xc3=\xec\xe3\x186\x05\x9al\x9b\xe0Z\xef\xc3\x18\x90\x13\t\x95o\xf9\xbf\xbc\xae?\x9d\xc7\xda#\xfaP:S\x8d_M\x98V\x92\xb0\x85\xa3\x9f\xc3-\x06\xa0h}\x16\x16\xc0\x83\x06\xf1\xb8z/\x85@\x91\r\xea\xd1\x13\xfdX\xb4Dkg{\xcf5(\xa7>2K\xcdg\xcbl\x9f\xdeO\xcc\x97I\xa8=\x9a\xab.,Q\x15ZY\x92UI)\x16nS\xb8}\x9a\x18\xe7\xe2t,\x1b\xe1\xa9\xf1&!?\x14\xe3\xf1{\xec\x0c$\xb6\x1d\x8f\xcc\xe2\xdbOj1\x9a>\xabo\xc2At\xfb\xf0\xc9\xa6\'8\xa8\xa7I\x1a\xe3\xf6\xd0m\x9c\x82\xed\xcd\xb7\xa0YNJ\t\x91D\x87\x9fus\xc9\x82\xd3\x13$"\x9a\x02\x88\xd1\xce\xe6\x05\xae(H\xc9\x83`\xec\xe9JtR\x9f]\xfc\xa9\xe7\x0b\xfc\x91\xcad\x93TL\xe9\xfa\x17\x8dE\xedm\x19jO\xf3\x83\x87f\x1c2\xba[\xce"Lk\x88\x9bp\xba\xc43\xb0\xe0\x9c(\xd2\x94\x8c9\x1c\xf1\xfc&\x9c\xd3\xd6;xQ_\xfb\xc0\x93\xe9\xc6\xa9\xa9T\x87\x88_\xff\xb95\xca\x8eK\xe7\xa6\tT\xab\x19\x05K\xbd\xee`\xe6lI\xf0C\xe30Zt\xe4\xd0\x8bsT=\x85\xfa\xee\xe6lm\xc6\xa9\xc8\x17,E\xdb\xc7\xdb?\t\xaa\xeb\xff\x05T\xd7\xbf\x01\xd5 \xda\x08*\xd9\x03P_e\x17x\xdc\x12\xfd\xb5\x81\xc9\xc3X\x84)\x01\xb3\x05\x82\xca\x94\xcd\x97\xc5\xc2\xb0\xf1\xae\x8d%~\xe4\xb5Z8\x04\x08\xb2K>\xc9\xbbFSf\xff\xd5(dT3+\xc5~<M\xa4\x14\xae\x8b\xced\xb0\x9a\x17\xd2DR\xb2\x96\xffd\xdd_{\xc0\xbf48\x86\x89\xfe*l\xee\x0et\xa6^Ua\xc2\x86\x8b\x98\x1aR\x86\xaaw\xd0"H\t#\xeb\x12\xbf\xa9U}\xcb\xd8co\xff\x95\xf4p\xfd\xe7\x83\xa8\xf8\xab\xec\xfd\x12\xed!Z\xde\x9bD\x1a\xae\x1em\x08\xf9\xd75\x0c\x98\xb1\xa0\x04\xf1\x9a\xfa\xe3\x9b\xb5\xcd\xc9\x80\'\x96\x87\xc2E\xc3\xb1@<\xfc\xbdd\xa8,\x0c\x07\x1b\xd2\x97\xa8\xfar\xc6.\x8f\t\xe2\xc1\xa5\xb7\xdf\xa6e"\xff\xdd\xc1\xa0\xed\xd0]=\xb0C5=\x13s\xde\xf555\x15\xbb}\xa7\xfax/\x98\x9d\xb3\xc7\x9b\xdf\xe2)S\x9480\xb8\x8a\xc5\xdb\x99\x13\xf9"\xe4\x0f\x9aI\xfc\x97t\x8c\xff\xa3\tNWg\xf7\xe7\xa8\x7fr\xa2=\x83\xe6\x9b^~,L\xa8]\x1aow\x94*R\xa3`\xc5Q41M\x1dq\x16*\x1f\xa2\n\x9d\x87MQu\x8el\x88\x9c\x15\xfd\xeaF\x9eUH\\\xde\xc4/\xf5\xb1\xc6\x0b7\xd4-j\x87\xe04\xf9{\x90xwk\x87\xa9\x06J\xd1\xf8A\x0e\xd0\x8f\x0c\x9c\x06q\x05/\xa2\xe4{\x81\xd6\x0eB\xc92\x19#\x95w3LC\xf4\xab\x12\x00\xa4\x12\xf8~\x85\x87\xfeG\xbe\xad:nZI\xc6\r\x019d\xda2q\x9b\x8c \x8e\xa6\x8f\xfb+\x7f\xbdx\xf6{\x9a[6~# =\x9bT\x9e\x15\xdc\x80h\xa7\xe41\x9d8q\xfaL\xa2;3\xac}g\xfcw\xacU\xadr\xe9i\x90\xaezw$\xcf^\xe4\xf5"\xf5\xda\x90\xab%W\x84\xbf\x15h\xfa\xa1\x8f|\xcdFo\x96\xe6f\xaa\xe5\x17\x94\xc3|\x1cR\xc8\xb6JR\x84\x06\x85?\xc7\x14,[Q\x99/o\xa6o\xe2\x9f\x98m\x919\xbf\xefdK\xe8\x90\xe7n\xc1\x98\x7f\x01J\xaf~D').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['music']
REMOTE_MAX_IMAGES = 0

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'Kaggle CUDA GPU 不可用；請啟用 GPU 並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [0]
        else:
            selected_gpus = [0, 1]
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != 'cuda:0':
                return False, 'Qwen Transformer 沒有設定為 cuda:0。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, 0), ('HeartCodec', pipeline.codec, 1)):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 8 節完成：選用 FIFO／ngrok API 已啟動。')


9. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_HeartMuLa_音樂生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 9 節完成：API Schema 和 client 說明已顯示。')
